In [ ]:
# Librerie di sistema e utilità
import os
import sys
import platform
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Librerie per elaborazione audio
import librosa
import librosa.display

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
from torchvision import transforms
import timm

# Metriche
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Ignoriamo i warning
warnings.filterwarnings("ignore")

print("Librerie importate con successo!")
print(f"PyTorch versione: {torch.__version__}")
print(f"timm versione: {timm.__version__}")
print(f"Python versione: {platform.python_version()}")

In [ ]:
import shutil
import os

clear_working_dir = True

working_dir = '/kaggle/working/'

if clear_working_dir:
    for filename in os.listdir(working_dir):
        file_path = os.path.join(working_dir, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)  # elimina file o link
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)  # elimina directory
        except Exception as e:
            print(f'Errore durante la rimozione di {file_path}: {e}')
    print(f"Tutti i file in {working_dir} sono stati rimossi.")
else:
    print("Pulizia disabilitata (clear_working_dir = False)")

In [ ]:
class Config:
    def __init__(self):
        
        # Imposta i percorsi di base in base all'ambiente
        self.COMPETITION_NAME = "birdclef-2025"
        self.BASE_DIR = f"/kaggle/input/{self.COMPETITION_NAME}"
        self.OUTPUT_DIR = "/kaggle/working"
        self.MODELS_DIR = "/kaggle/input"  # Per i modelli pre-addestrati
            
        # Imposta subito i percorsi derivati per l'ambiente Kaggle
        self._setup_derived_paths()
            
        
        # Parametri per il preprocessing audio
        self.SR = 32000      # Sample rate
        self.DURATION = 5    # Durata dei clip in secondi
        self.N_MELS = 224    # Numero di bande Mel
        self.N_FFT = 2048    # Dimensione finestra FFT
        self.HOP_LENGTH = 512  # Hop length per STFT
        self.FMIN = 48       # Frequenza minima per lo spettrogramma Mel
        self.FMAX = 16000    # Frequenza massima
        self.POWER = 2       # Esponente per calcolo spettrogramma
            
        # Parametri per il training
        self.BATCH_SIZE = 96  
        self.EPOCHS = 15     
        self.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        self.NUM_WORKERS = 4  

        # Parametri per inference/submission
        self.TEST_CLIP_DURATION = 5  # Durata dei segmenti per la predizione (secondi)
        self.N_CLASSES = 0 

    def _setup_derived_paths(self):
        """Imposta i percorsi derivati basati su BASE_DIR"""
        self.TRAIN_AUDIO_DIR = os.path.join(self.BASE_DIR, "train_audio")
        self.TEST_SOUNDSCAPES_DIR = os.path.join(self.BASE_DIR, "test_soundscapes")
        self.TRAIN_CSV_PATH = os.path.join(self.BASE_DIR, "train.csv")
        self.TAXONOMY_CSV_PATH = os.path.join(self.BASE_DIR, "taxonomy.csv") 
        self.SAMPLE_SUB_PATH = os.path.join(self.BASE_DIR, "sample_submission.csv")

In [ ]:
config = Config()

# Stampa percorsi aggiornati
print(f"\nPercorso file CSV di training: {config.TRAIN_CSV_PATH}")
print(f"Percorso directory audio di training: {config.TRAIN_AUDIO_DIR}")

In [ ]:
# Crea una singola istanza della trasformazione MelSpectrogram da riutilizzare
mel_transform = T.MelSpectrogram(
    sample_rate=config.SR,
    n_fft=config.N_FFT,
    win_length=None,
    hop_length=config.HOP_LENGTH,
    f_min=config.FMIN,
    f_max=config.FMAX,
    n_mels=config.N_MELS,
    window_fn=torch.hann_window,
    power=config.POWER,
    normalized=False,
    onesided=True,
    norm="slaney",
    mel_scale="slaney"
)

# Funzione di conversione a dB e normalizzazione
def amplitude_to_db(spectrogram):
    """Converti spettrogramma in scala dB e normalizza tra 0-1"""
    # Converti in dB
    spectrogram_db = 10.0 * torch.log10(torch.clamp(spectrogram, min=1e-10))
    
    # Normalizza
    min_val = torch.min(spectrogram_db)
    max_val = torch.max(spectrogram_db)
    if max_val > min_val:
        return (spectrogram_db - min_val) / (max_val - min_val)
    else:
        return torch.zeros_like(spectrogram_db)

In [ ]:
# I parametri principali sono già definiti nella classe Config
# Verifichiamo l'esistenza delle directory e creiamo quelle necessarie per l'output

def setup_output_directories():
    """
    Configura le directory per l'output del progetto.
    
    Returns:
        dict: Dictionary con i percorsi delle directory di output
    """
    # Directory principale di output
    output_dir = config.OUTPUT_DIR
    
    # Sotto-directory per diversi tipi di output
    dirs = {
        'checkpoints': os.path.join(output_dir, 'checkpoints'),
        'tensorboard': os.path.join(output_dir, 'tensorboard_logs'),
        'predictions': os.path.join(output_dir, 'predictions'),
        'submissions': os.path.join(output_dir, 'submissions'),
        'visualizations': os.path.join(output_dir, 'visualizations'),
    }
    
    # Crea tutte le directory
    for dir_name, dir_path in dirs.items():
        os.makedirs(dir_path, exist_ok=True)
        print(f"Directory '{dir_name}' creata/verificata in: {dir_path}")
    
    return dirs

# Configura le directory di output
output_dirs = setup_output_directories()

# Memorizziamo i parametri di configurazione principali per l'addestramento
print("\nParametri di configurazione principali:")
print(f"- Sample rate: {config.SR} Hz")
print(f"- Durata clip audio: {config.DURATION} secondi")
print(f"- Numero bande Mel: {config.N_MELS}")
print(f"- Dimensione FFT: {config.N_FFT}")
print(f"- Hop length: {config.HOP_LENGTH}")
print(f"- Device: {config.DEVICE}")
print(f"- Batch size: {config.BATCH_SIZE}")
print(f"- Epoche: {config.EPOCHS}")

In [ ]:
# Caricamento dei metadati
def load_metadata():
    """
    Carica e prepara i metadati dal file CSV di training.
    
    Returns:
        tuple: training_df, all_species, labels_one_hot
    """
    print(f"Caricamento metadati da: {config.TRAIN_CSV_PATH}")
    train_df = pd.read_csv(config.TRAIN_CSV_PATH)
    sample_sub_df = pd.read_csv(config.SAMPLE_SUB_PATH)
    
    # Estrai tutte le etichette uniche
    train_primary_labels = train_df['primary_label'].unique()
    train_secondary_labels = set([lbl for sublist in train_df['secondary_labels'].apply(eval) 
                                 for lbl in sublist if lbl])
    submission_species = sample_sub_df.columns[1:].tolist() 
    
    # Combina tutte le possibili etichette
    all_species = sorted(list(set(train_primary_labels) | train_secondary_labels | set(submission_species)))
    N_CLASSES = len(all_species)
    config.N_CLASSES = N_CLASSES  
    
    print(f"Numero totale di specie trovate: {N_CLASSES}")
    print(f"Prime 10 specie: {all_species[:10]}")
    
    # Crea mappatura etichette-indici
    species_to_int = {species: i for i, species in enumerate(all_species)}
    int_to_species = {i: species for species, i in species_to_int.items()}
    
    # Aggiungi indici numerici al dataframe
    train_df['primary_label_int'] = train_df['primary_label'].map(species_to_int)
    
    # Prepara target multi-etichetta
    mlb = MultiLabelBinarizer(classes=all_species)
    mlb.fit(None)  
    
    def get_multilabel(row):
        labels = eval(row['secondary_labels'])  
        labels.append(row['primary_label'])
        return list(set(labels))  
    
    train_df['all_labels'] = train_df.apply(get_multilabel, axis=1)
    train_labels_one_hot = mlb.transform(train_df['all_labels'])
    
    print(f"Forma delle etichette one-hot: {train_labels_one_hot.shape}")
    
    return train_df, all_species, train_labels_one_hot, species_to_int, int_to_species

# Carica i metadati
train_df, all_species, train_labels_one_hot, species_to_int, int_to_species = load_metadata()


def augment_rare_classes(train_df, labels_one_hot, rare_threshold=10, target_samples=30):
    """
    Applica oversampling con data augmentation alle classi rare.
    
    Args:
        train_df: DataFrame con i metadati
        labels_one_hot: Array di etichette one-hot
        rare_threshold: Soglia per definire una classe rara
        target_samples: Numero minimo di campioni da raggiungere per ogni classe
    
    Returns:
        tuple: DataFrame e labels_one_hot bilanciati con classi rare sovracampionate
    """
    print("\n=== Augmentazione delle Classi Rare ===")
    
    # Conta esempi per ogni classe
    class_counts = train_df['primary_label'].value_counts()
    
    # Identifica classi rare
    rare_classes = class_counts[class_counts <= rare_threshold].index.tolist()
    print(f"Classi identificate come rare (≤{rare_threshold} esempi): {len(rare_classes)}")
    
    augmented_rows = []
    augmented_labels = []
    
    # Contatore per il monitoraggio
    augmentation_count = 0
    
    # Per ogni classe rara
    for cls in rare_classes:
        # Filtra esempi di questa classe
        class_df = train_df[train_df['primary_label'] == cls]
        class_indices = class_df.index.tolist()
        current_count = len(class_df)
        
        # Determina quanti esempi aggiuntivi servono
        n_to_add = max(0, target_samples - current_count)
        
        # Se serve aggiungere campioni
        if n_to_add > 0:
            print(f"Classe {cls}: aggiunta di {n_to_add} esempi augmentati (da {current_count} a {target_samples})")
            
            # Cicla finché non abbiamo abbastanza esempi
            for _ in range(n_to_add):
                # Seleziona un esempio casuale da replicare
                orig_idx = np.random.choice(class_indices)
                orig_row = train_df.iloc[orig_idx].copy()
                
                # Marca questa riga come augmentata
                orig_row['is_augmented'] = 1
                
                # Aggiungi alle liste
                augmented_rows.append(orig_row)
                augmented_labels.append(labels_one_hot[orig_idx])
                augmentation_count += 1
    
    # Se abbiamo augmentato qualcosa
    if augmented_rows:
        # Crea nuovo DataFrame con gli esempi originali + augmentati
        augmented_df = pd.concat([train_df, pd.DataFrame(augmented_rows)], ignore_index=True)
        
        # Aggiungi la colonna is_augmented se non esiste
        if 'is_augmented' not in augmented_df.columns:
            augmented_df.insert(len(augmented_df.columns), 'is_augmented', 0)
        
        # Aggiorna anche le etichette one-hot
        augmented_labels_one_hot = np.vstack([labels_one_hot, np.array(augmented_labels)])
        
        print(f"Dataset originale: {len(train_df)} esempi")
        print(f"Dataset augmentato: {len(augmented_df)} esempi (+{augmentation_count} esempi augmentati)")
        
        # Verifica distribuzione finale
        final_counts = augmented_df['primary_label'].value_counts()
        rare_with_few = sum(1 for c in final_counts.values if c < rare_threshold)
        print(f"Classi con ancora meno di {rare_threshold} esempi: {rare_with_few}")
        
        return augmented_df, augmented_labels_one_hot
    else:
        return train_df, labels_one_hot

In [ ]:
# Suddividi i dati in training e validation
def split_data(train_df, labels_one_hot, test_size=0.2, random_state=42):
    """
    Suddivide il dataset in set di training e validation mantenendo la distribuzione delle classi.
    
    Args:
        train_df: DataFrame con i metadati
        labels_one_hot: Array di etichette one-hot
        test_size: Percentuale dei dati da usare per validation
        random_state: Seed per riproducibilità
        
    Returns:
        tuple: X_train_df, X_val_df, y_train_one_hot, y_val_one_hot
    """
    # Indici per lo split STRATIFICATO basato sulle etichette primarie
    train_indices, val_indices = train_test_split(
        range(len(train_df)),
        test_size=test_size,
        random_state=random_state,
        stratify=train_df['primary_label']  
    )
    
    # Crea i dataframe e gli array di etichette splittati
    X_train_df = train_df.iloc[train_indices].reset_index(drop=True)
    X_val_df = train_df.iloc[val_indices].reset_index(drop=True)
    
    y_train_one_hot = labels_one_hot[train_indices]
    y_val_one_hot = labels_one_hot[val_indices]
    
    print(f"Dimensioni Training Set: {X_train_df.shape}, Etichette: {y_train_one_hot.shape}")
    print(f"Dimensioni Validation Set: {X_val_df.shape}, Etichette: {y_val_one_hot.shape}")
    
    # Verifica presenza di tutte le classi nei set
    train_classes = set(X_train_df['primary_label'].unique())
    val_classes = set(X_val_df['primary_label'].unique())
    all_dataset_classes = set(train_df['primary_label'].unique())
    
    print(f"Classi totali nel dataset: {len(all_dataset_classes)}")
    print(f"Classi nel training set: {len(train_classes)}")
    print(f"Classi nel validation set: {len(val_classes)}")
    
    # Verifica classi mancanti
    missing_in_train = all_dataset_classes - train_classes
    missing_in_val = all_dataset_classes - val_classes
    
    if missing_in_train:
        print(f"ATTENZIONE: {len(missing_in_train)} classi mancanti nel training set!")
    if missing_in_val:
        print(f"ATTENZIONE: {len(missing_in_val)} classi mancanti nel validation set!")
    
    return X_train_df, X_val_df, y_train_one_hot, y_val_one_hot



# Applica oversampling mirato per le classi rare prima dello split
train_df, train_labels_one_hot = augment_rare_classes(
    train_df, 
    train_labels_one_hot,
    rare_threshold=35,  # Considera rare le classi con ≤10 esempi
    target_samples=40   # Porta ogni classe ad avere almeno 40 esempi
)

# Ora procedi con lo split train/val stratificato
X_train_df, X_val_df, y_train_one_hot, y_val_one_hot = split_data(train_df, train_labels_one_hot)

X_test_df = None
y_test_one_hot = None

In [ ]:
def load_and_preprocess_audio_torch(file_path, target_sr=config.SR, duration=config.DURATION, 
                                    segment_position='center', random_segment=False):
    """
    Carica un file audio, estrae un segmento specifico e lo converte in spettrogramma Mel usando PyTorch.
    """
    try:
        # Carica il file audio con torchaudio
        waveform, sr = torchaudio.load(file_path)
        
        if waveform.shape[0] > 1:
            waveform = waveform[0:1]
        
        # Ricampiona se necessario
        if sr != target_sr:
            resampler = T.Resample(sr, target_sr)
            waveform = resampler(waveform)
        
        # Calcola la lunghezza target in campioni
        target_len = int(target_sr * duration)
        total_len = waveform.shape[1]
        
        # Gestisci clip troppo corte
        if total_len < target_len:
            # Ripeti l'audio per raggiungere la lunghezza target
            n_repeat = (target_len // total_len) + 1
            waveform = waveform.repeat(1, n_repeat)
            total_len = waveform.shape[1]
        
        # Seleziona il segmento
        if random_segment and total_len > target_len:
            # Estrai segmento casuale
            max_start_idx = total_len - target_len
            start_idx = torch.randint(0, max_start_idx, (1,)).item()
        else:
            # Usa le posizioni predefinite
            if segment_position == 'start':
                start_idx = int(total_len * 0.2)
                if start_idx + target_len > total_len:
                    start_idx = max(0, total_len - target_len)
            elif segment_position == 'end':
                end_point = int(total_len * 0.8)
                start_idx = max(0, end_point - target_len)
            else:  # 'center' (default)
                start_idx = max(0, int(total_len / 2 - target_len / 2))
        
        # Estrai il segmento
        waveform = waveform[:, start_idx:start_idx + target_len]
        
        # Padda se necessario
        if waveform.shape[1] < target_len:
            padding = target_len - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        
        # Applica la trasformazione Mel
        mel_spectrogram = mel_transform(waveform)
        
        # Converti in scala dB e normalizza
        log_mel_spec = amplitude_to_db(mel_spectrogram)

        # Ridimensiona per il modello EfficientNet
        resize_transform = transforms.Resize((224, 224), 
            interpolation=transforms.InterpolationMode.BICUBIC)
        log_mel_spec = resize_transform(log_mel_spec)
        
        return log_mel_spec
        
    except Exception as e:
        print(f"Errore nell'elaborazione di {file_path}: {e}")
        # Crea uno spettrogramma vuoto
        time_steps = int(target_sr * duration / config.HOP_LENGTH) + 1
        return torch.zeros((1, config.N_MELS, time_steps), dtype=torch.float32)

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, labels_one_hot):
        """Dataset per la validazione che estrae solo il segmento centrale."""
        self.df = df
        self.audio_dir = audio_dir
        self.labels = labels_one_hot
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['filename']
        file_path = os.path.join(self.audio_dir, filename)
        
        if not os.path.exists(file_path):
            print(f"Attenzione: File non trovato in {file_path}.")
            time_steps = int(config.SR * config.DURATION / config.HOP_LENGTH) + 1
            dummy_spec = torch.zeros((1, config.N_MELS, time_steps), dtype=torch.float32)
            dummy_label = torch.zeros(config.N_CLASSES, dtype=torch.float32)
            return dummy_spec, dummy_label
            
        # Carica e preprocessa l'audio con la funzione PyTorch
        mel_spec_tensor = load_and_preprocess_audio_torch(file_path, segment_position='center')
        
        # Ottieni le etichette
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.float32)
            
        return mel_spec_tensor, label_tensor

# Crea dataset di validation
val_dataset = BirdDataset(X_val_df, config.TRAIN_AUDIO_DIR, y_val_one_hot)

# Crea dataloader di validation
val_loader = DataLoader(
    val_dataset, 
    batch_size=config.BATCH_SIZE, 
    shuffle=False,
    num_workers=config.NUM_WORKERS, 
    pin_memory=True
)

print(f"Numero di batch di validation: {len(val_loader)}")

In [ ]:
class EfficientNetBirdClassifier(nn.Module):
    def __init__(self, num_classes=config.N_CLASSES, pretrained=False, model_name='efficientnet_b0'):
        super(EfficientNetBirdClassifier, self).__init__()
        
        self.efficientnet = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0 
        )
        
        # Ottieni la dimensione dell'output del feature extractor
        if hasattr(self.efficientnet, 'num_features'):
            classifier_in_features = self.efficientnet.num_features
        elif hasattr(self.efficientnet, 'classifier'):
            classifier_in_features = self.efficientnet.classifier.in_features
        else:
            # Valore predefinito per EfficientNet-B0
            classifier_in_features = 1280
        
        # Sostituisci il classificatore semplice con una MLP con dropout
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),  # Primo dropout significativo
            nn.Linear(classifier_in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),  # Secondo dropout più leggero
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        # Se l'input è un'immagine a 1 canale, replicala su 3 canali
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        
        # Passa l'input attraverso il backbone per ottenere le features
        features = self.efficientnet(x)
        
        # Passa le feature attraverso il classificatore
        output = self.classifier(features)
        
        return output

In [ ]:
def load_model(model_path, device=config.DEVICE, model_name='efficientnet_b0'):
    """
    Carica un modello pre-addestrato da un file checkpoint.
    
    Args:
        model_path: Path del checkpoint del modello
        device: Device su cui caricare il modello ('cuda' o 'cpu')
        model_name: Nome del modello da caricare

    Returns:
        model: Modello caricato
    """
    # Inizializza il modello
    model = EfficientNetBirdClassifier(
        num_classes=config.N_CLASSES,
        pretrained=False,
        model_name=model_name
    ).to(device)
    
    # Carica i pesi
    try:
        print(f"Caricamento modello da {model_path}...")
        checkpoint = torch.load(model_path, map_location=device)
        
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"Modello caricato con successo (epoca: {checkpoint.get('epoch', 'N/A')})")
        else:
            model.load_state_dict(checkpoint)
            print("Modello caricato con successo")
        
        model.eval() 
    except Exception as e:
        print(f"Errore nel caricamento del modello: {e}")
        raise
    
    return model

In [ ]:
def get_model_predictions(model, dataloader, device=config.DEVICE):
    """
    Ottiene le predizioni e i ground truth per un modello su un dataloader.
    
    Returns:
        tuple: y_true, y_pred_prob, y_pred_binary
    """
    model.eval()
    all_labels = []
    all_predictions_prob = []
    all_predictions_binary = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc=f"Valutazione modello"):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            probs = torch.sigmoid(outputs)
            binary_preds = (probs > 0.5).float()
            
            all_labels.extend(labels.cpu().numpy())
            all_predictions_prob.extend(probs.cpu().numpy())
            all_predictions_binary.extend(binary_preds.cpu().numpy())
    
    return np.array(all_labels), np.array(all_predictions_prob), np.array(all_predictions_binary)

def calculate_metrics(y_true, y_pred, y_pred_prob):
    """
    Calcola le metriche di performance per un modello.
    
    Returns:
        dict: Dizionario con le metriche
    """
    metrics = {}
    
    # Calcola precision, recall e F1 per ogni classe
    metrics['precision_per_class'] = precision_score(y_true, y_pred, average=None, zero_division=0)
    metrics['recall_per_class'] = recall_score(y_true, y_pred, average=None, zero_division=0)
    metrics['f1_per_class'] = f1_score(y_true, y_pred, average=None, zero_division=0)
    
    metrics['auc_per_class'] = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) > 1: 
            try:
                auc = roc_auc_score(y_true[:, i], y_pred_prob[:, i])
                metrics['auc_per_class'].append(auc)
            except:
                metrics['auc_per_class'].append(float('nan'))
        else:
            metrics['auc_per_class'].append(float('nan'))
    metrics['auc_per_class'] = np.array(metrics['auc_per_class'])
    
    # Calcola metriche medie
    metrics['precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['recall_macro'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['f1_macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['precision_micro'] = precision_score(y_true, y_pred, average='micro', zero_division=0)
    metrics['recall_micro'] = recall_score(y_true, y_pred, average='micro', zero_division=0)
    metrics['f1_micro'] = f1_score(y_true, y_pred, average='micro', zero_division=0)
    
    # Calcola metriche per esempio
    metrics['precision_per_sample'] = np.mean([(y_pred[i] * y_true[i]).sum() / max(y_pred[i].sum(), 1) 
                                           for i in range(len(y_true))])
    metrics['recall_per_sample'] = np.mean([(y_pred[i] * y_true[i]).sum() / max(y_true[i].sum(), 1) 
                                        for i in range(len(y_true))])
    
    # Calcola errore per esempio per il t-test
    metrics['error_per_sample'] = np.mean((y_pred_prob - y_true) ** 2, axis=1)
    
    return metrics

def compare_models_with_ttest(metrics1, metrics2):
    """
    Confronta due modelli utilizzando il t-test accoppiato.
    
    Args:
        metrics1: Metriche del primo modello
        metrics2: Metriche del secondo modello
        
    Returns:
        dict: Risultati del t-test
    """
    results = {}
    
    # Confronta gli errori per esempio
    t_stat, p_value = stats.ttest_rel(metrics1['error_per_sample'], metrics2['error_per_sample'])
    
    # Calcola la differenza media
    mean_diff = metrics1['error_per_sample'].mean() - metrics2['error_per_sample'].mean()
    
    results['t_statistic'] = t_stat
    results['p_value'] = p_value
    results['mean_difference'] = mean_diff
    results['better_model'] = "Modello 1" if mean_diff < 0 else "Modello 2"
    results['is_significant'] = p_value < 0.05
    
    return results

def visualize_comparison(metrics1, metrics2, ttest_results, model1_name, model2_name):
    """
    Visualizza il confronto tra due modelli.
    """
    # Imposta lo stile dei grafici
    plt.style.use('seaborn-v0_8')
    
    # Crea una figura con 2 subplot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Confronto delle metriche macro
    metrics_to_compare = ['precision_macro', 'recall_macro', 'f1_macro', 'precision_micro', 'recall_micro', 'f1_micro']
    x = np.arange(len(metrics_to_compare))
    width = 0.35
    
    # Estrai valori per il primo modello
    values1 = [metrics1[metric] for metric in metrics_to_compare]
    # Estrai valori per il secondo modello
    values2 = [metrics2[metric] for metric in metrics_to_compare]
    
    # Crea il grafico a barre
    bars1 = ax1.bar(x - width/2, values1, width, label=model1_name)
    bars2 = ax1.bar(x + width/2, values2, width, label=model2_name)
    
    ax1.set_title('Confronto delle Metriche di Performance', fontsize=15)
    ax1.set_xticks(x)
    ax1.set_xticklabels([m.replace('_', ' ').title() for m in metrics_to_compare], rotation=45)
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.7)
    
    # Aggiungi i valori sopra le barre
    for i, v in enumerate(values1):
        ax1.text(i - width/2, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    for i, v in enumerate(values2):
        ax1.text(i + width/2, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Grafico dell'errore medio per esempio
    error1 = metrics1['error_per_sample'].mean()
    error2 = metrics2['error_per_sample'].mean()
    
    ax2.bar([0, 1], [error1, error2], color=['#1f77b4', '#ff7f0e'])
    ax2.set_title('Errore Medio Quadratico per Esempio (MSE)', fontsize=15)
    ax2.set_xticks([0, 1])
    ax2.set_xticklabels([model1_name, model2_name])
    ax2.grid(True, linestyle='--', alpha=0.7)
    
    # Aggiungi i valori sopra le barre
    ax2.text(0, error1 + 0.001, f'{error1:.5f}', ha='center', va='bottom')
    ax2.text(1, error2 + 0.001, f'{error2:.5f}', ha='center', va='bottom')
    
    # Aggiungi risultato del t-test
    title = f"Risultato T-test: {'Significativo' if ttest_results['is_significant'] else 'Non significativo'}"
    fig.suptitle(title + f" (p-value: {ttest_results['p_value']:.5f})", fontsize=16)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

In [ ]:
model1_path = "/kaggle/input/efficientnet_jft_45/pytorch/default/1/birdclef_efficientNETJFT_30Epoch_AugPaper.pth"
model2_path = "/kaggle/input/efficientnet_jft_augpaper30epochs/pytorch/default/1/birdclef_efficientNETJFT_30Epoch_AugPaper_best (1).pth"

model1_name = "EfficientNet con JFT e NS 45 epochs"
model2_name = "EfficientNet con JFT e NS 30 epochs"

# Ottieni predizioni per il primo modello
print(f"\nOttenimento predizioni per {model1_name}...")
y_true1, y_pred_prob1, y_pred_binary1 = get_model_predictions(model1, val_loader)

# Ottieni predizioni per il secondo modello
print(f"\nOttenimento predizioni per {model2_name}...")
y_true2, y_pred_prob2, y_pred_binary2 = get_model_predictions(model2, val_loader)

# Calcola le metriche per entrambi i modelli
metrics1 = calculate_metrics(y_true1, y_pred_binary1, y_pred_prob1)
metrics2 = calculate_metrics(y_true2, y_pred_binary2, y_pred_prob2)

# Esegui il t-test
ttest_results = compare_models_with_ttest(metrics1, metrics2)

# Visualizza i risultati del confronto
visualize_comparison(metrics1, metrics2, ttest_results, model1_name, model2_name)

# Stampa i risultati dettagliati
print("\n=== RISULTATI DEL CONFRONTO ===")
print(f"Modello 1: {model1_name}")
print(f"Modello 2: {model2_name}")

# Stampa metriche principali
print("\nMetriche principali:")
for metric in ['precision_macro', 'recall_macro', 'f1_macro']:
    print(f"{metric.replace('_', ' ').title()}:")
    print(f"  - {model1_name}: {metrics1[metric]:.4f}")
    print(f"  - {model2_name}: {metrics2[metric]:.4f}")
    print(f"  - Differenza: {metrics1[metric] - metrics2[metric]:.4f}")

# Stampa risultati t-test
print("\nRisultati del T-test:")
print(f"- T-statistic: {ttest_results['t_statistic']:.4f}")
print(f"- P-value: {ttest_results['p_value']:.5f}")
print(f"- Differenza media nell'errore: {ttest_results['mean_difference']:.5f}")
print(f"- Modello migliore: {ttest_results['better_model']}")
print(f"- Differenza statisticamente significativa: {'Sì' if ttest_results['is_significant'] else 'No'}")

# Visualizzazione delle distribuzioni degli errori
plt.figure(figsize=(12, 6))
sns.histplot(metrics1['error_per_sample'], kde=True, label=model1_name, alpha=0.6)
sns.histplot(metrics2['error_per_sample'], kde=True, label=model2_name, alpha=0.6)
plt.title('Distribuzione degli Errori per Esempio', fontsize=15)
plt.xlabel('Errore Quadratico Medio')
plt.ylabel('Frequenza')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()